In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Phase 3: Classification — FD002 (Corrected)\n",
    "This notebook implements the classification phase for the FD002 dataset with fixes for data leakage and proper evaluation."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 0: Setup — Config, Folders, and Random Seed"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "import numpy as np\n",
    "import pandas as pd\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score\n",
    "from sklearn.ensemble import RandomForestClassifier\n",
    "from sklearn.svm import SVC\n",
    "from sklearn.linear_model import LogisticRegression\n",
    "from xgboost import XGBClassifier\n",
    "from sklearn.metrics import classification_report, confusion_matrix\n",
    "from sklearn.preprocessing import StandardScaler\n",
    "import joblib\n",
    "\n",
    "# Manual config — assuming we're inside notebooks/phase3, go two levels up\n",
    "BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))\n",
    "FIGURES_DIR = os.path.join(BASE_DIR, 'figures', 'phase3')\n",
    "MODELS_DIR = os.path.join(BASE_DIR, 'models')\n",
    "REPORT_DIR = os.path.join(BASE_DIR, 'report')\n",
    "CLASSIF_REPORT_DIR = os.path.join(REPORT_DIR, 'classification', 'FD002')  # Subfolder for FD002\n",
    "\n",
    "RANDOM_STATE = 42  # Global seed\n",
    "\n",
    "# Set consistent plotting aesthetics\n",
    "sns.set_style('whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (10, 6)\n",
    "plt.rcParams['axes.labelsize'] = 12\n",
    "plt.rcParams['axes.titlesize'] = 14\n",
    "\n",
    "# Set reproducibility\n",
    "np.random.seed(RANDOM_STATE)\n",
    "\n",
    "# Create required folders\n",
    "os.makedirs(FIGURES_DIR, exist_ok=True)\n",
    "os.makedirs(MODELS_DIR, exist_ok=True)\n",
    "os.makedirs(CLASSIF_REPORT_DIR, exist_ok=True)\n",
    "\n",
    "# Confirm setup\n",
    "print(f\"Figures directory ready: {FIGURES_DIR}\")\n",
    "print(f\"Models directory ready: {MODELS_DIR}\")\n",
    "print(f\"Classification report directory ready: {CLASSIF_REPORT_DIR}\")\n",
    "print(f\"Random seed set: {RANDOM_STATE}\")\n",
    "print(\"✅ Step 0 complete — Phase 3 (FD002) setup initialized.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 1: Load Cleaned + Final Cluster-Labeled FD002 Dataset"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# File path to labeled FD002 dataset\n",
    "DATA_FILE_FD002 = os.path.join(BASE_DIR, 'data', 'corrected_clustered_train_FD002.csv')\n",
    "\n",
    "# Load the dataset\n",
    "df_fd002 = pd.read_csv(DATA_FILE_FD002)\n",
    "print(f\"FD002 dataset loaded from: {DATA_FILE_FD002}\")\n",
    "\n",
    "# --- Validate structure ---\n",
    "assert 'final_stage' in df_fd002.columns, \"'final_stage' column missing in dataset.\"\n",
    "\n",
    "# Show head of the dataset\n",
    "print(\"\\nPreview — First 5 rows:\")\n",
    "print(df_fd002.head())\n",
    "\n",
    "# Dataset information\n",
    "print(\"\\nℹ Dataset Info:\")\n",
    "print(df_fd002.info())\n",
    "\n",
    "# Check for missing values\n",
    "nulls = df_fd002.isnull().sum()\n",
    "if nulls.any():\n",
    "    print(\"\\nMissing values found:\")\n",
    "    print(nulls[nulls > 0])\n",
    "else:\n",
    "    print(\"\\nNo missing values.\")\n",
    "\n",
    "# Class distribution (0–4 stages)\n",
    "print(\"\\nFinal Stage Distribution (Count per Class):\")\n",
    "print(df_fd002['final_stage'].value_counts().sort_index())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 2: Feature and Target Setup"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Feature and target setup\n",
    "df = df_fd002.copy()\n",
    "\n",
    "# Dynamically select all usable sensor columns\n",
    "selected_features = [col for col in df.columns if col.startswith(\"sensor_\")]\n",
    "\n",
    "# Extract features and target\n",
    "X_fd002 = df[selected_features]\n",
    "y_fd002 = df['final_stage']\n",
    "\n",
    "# Display shapes and stats\n",
    "print(f\"✅ Selected features: {len(selected_features)} sensors → {selected_features}\")\n",
    "print(\"🔹 Input Features (X) shape:\", X_fd002.shape)\n",
    "print(\"🔹 Target (y) shape:\", y_fd002.shape)\n",
    "\n",
    "# Basic stats\n",
    "print(\"\\n📊 Feature Stats Summary:\")\n",
    "print(X_fd002.describe())\n",
    "\n",
    "# Target class distribution\n",
    "print(\"\\n📊 Class Distribution in y:\")\n",
    "print(y_fd002.value_counts())"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 3: Stratified Train-Test Split"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Stratified train-test split (80% train, 20% test)\n",
    "X_train, X_test, y_train, y_test = train_test_split(\n",
    "    X_fd002, y_fd002, test_size=0.2, stratify=y_fd002, random_state=RANDOM_STATE\n",
    ")\n",
    "\n",
    "# Further split train into train and validation (80% train, 20% validation of train set)\n",
    "X_train, X_val, y_train, y_val = train_test_split(\n",
    "    X_train, y_train, test_size=0.2, stratify=y_train, random_state=RANDOM_STATE\n",
    ")\n",
    "\n",
    "# Summary\n",
    "print(\"Train Set Shape:\", X_train.shape)\n",
    "print(\"Validation Set Shape:\", X_val.shape)\n",
    "print(\"Test Set Shape:\", X_test.shape)\n",
    "\n",
    "print(\"\\nTrain Final Stage Distribution:\")\n",
    "print(y_train.value_counts(normalize=True).sort_index().apply(lambda x: f\"{x:.2%}\"))\n",
    "\n",
    "print(\"\\nValidation Final Stage Distribution:\")\n",
    "print(y_val.value_counts(normalize=True).sort_index().apply(lambda x: f\"{x:.2%}\"))\n",
    "\n",
    "print(\"\\nTest Final Stage Distribution:\")\n",
    "print(y_test.value_counts(normalize=True).sort_index().apply(lambda x: f\"{x:.2%}\"))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 4: Preprocessing — Scaling"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize scaler\n",
    "scaler = StandardScaler()\n",
    "\n",
    "# Fit on training data and transform train, val, test\n",
    "X_train_scaled = scaler.fit_transform(X_train)\n",
    "X_val_scaled = scaler.transform(X_val)\n",
    "X_test_scaled = scaler.transform(X_test)\n",
    "\n",
    "print(\"✅ Data scaled using StandardScaler.\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 5: Train & Evaluate Random Forest Classifier"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Random Forest Model ---\n",
    "rf = RandomForestClassifier(\n",
    "    n_estimators=100,\n",
    "    max_depth=10,\n",
    "    class_weight='balanced',\n",
    "    random_state=RANDOM_STATE\n",
    ")\n",
    "\n",
    "# --- Cross-Validation ---\n",
    "cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)\n",
    "cv_scores = cross_val_score(rf, X_train_scaled, y_train, cv=cv, scoring='f1_weighted')\n",
    "print(\"RF 5-Fold CV F1 Scores:\", cv_scores)\n",
    "print(\"RF Mean F1 Score: {:.4f}\".format(cv_scores.mean()))\n",
    "\n",
    "# --- Train ---\n",
    "rf.fit(X_train_scaled, y_train)\n",
    "\n",
    "# --- Predict on Validation Set ---\n",
    "y_pred_val = rf.predict(X_val_scaled)\n",
    "\n",
    "# --- Evaluation ---\n",
    "print(\"\\nClassification Report (Validation Set):\")\n",
    "print(classification_report(y_val, y_pred_val))\n",
    "\n",
    "# --- Confusion Matrix ---\n",
    "labels = [0, 1, 2, 3, 4]\n",
    "cm = confusion_matrix(y_val, y_pred_val, labels=labels, normalize='true')\n",
    "plt.figure(figsize=(8, 6))\n",
    "sns.heatmap(cm, annot=True, fmt=\".2f\", cmap='Blues',\n",
    "            xticklabels=[f\"Stage {i}\" for i in labels],\n",
    "            yticklabels=[f\"Stage {i}\" for i in labels])\n",
    "plt.title(\"FD002 – Random Forest Confusion Matrix (Validation Set)\")\n",
    "plt.xlabel(\"Predicted\")\n",
    "plt.ylabel(\"Actual\")\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# --- Save Model ---\n",
    "model_path = os.path.join(MODELS_DIR, 'rf_classifier_fd002.pkl')\n",
    "joblib.dump(rf, model_path)\n",
    "print(f\"Model saved to: {model_path}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 6: Train & Evaluate SVM Classifier"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- SVM Model ---\n",
    "svm = SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=RANDOM_STATE)\n",
    "\n",
    "# --- Cross-Validation ---\n",
    "cv_scores = cross_val_score(svm, X_train_scaled, y_train, cv=cv, scoring='f1_weighted')\n",
    "print(\"SVM 5-Fold CV F1 Scores:\", cv_scores)\n",
    "print(\"SVM Mean F1 Score: {:.4f}\".format(cv_scores.mean()))\n",
    "\n",
    "# --- Train ---\n",
    "svm.fit(X_train_scaled, y_train)\n",
    "\n",
    "# --- Predict on Validation Set ---\n",
    "y_pred_val = svm.predict(X_val_scaled)\n",
    "\n",
    "# --- Evaluation ---\n",
    "print(\"\\nClassification Report (Validation Set):\")\n",
    "print(classification_report(y_val, y_pred_val))\n",
    "\n",
    "# --- Confusion Matrix ---\n",
    "cm = confusion_matrix(y_val, y_pred_val, labels=labels, normalize='true')\n",
    "plt.figure(figsize=(8, 6))\n",
    "sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',\n",
    "            xticklabels=[f\"Stage {l}\" for l in labels],\n",
    "            yticklabels=[f\"Stage {l}\" for l in labels])\n",
    "plt.title(\"FD002 – SVM Confusion Matrix (Validation Set)\")\n",
    "plt.xlabel(\"Predicted\")\n",
    "plt.ylabel(\"Actual\")\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# --- Save Model ---\n",
    "model_path = os.path.join(MODELS_DIR, 'svm_classifier_fd002.pkl')\n",
    "joblib.dump(svm, model_path)\n",
    "print(f\"Model saved to: {model_path}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 7: Train & Evaluate Logistic Regression Classifier"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- Logistic Regression Model ---\n",
    "lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)\n",
    "\n",
    "# --- Cross-Validation ---\n",
    "cv_scores = cross_val_score(lr, X_train_scaled, y_train, cv=cv, scoring='f1_weighted')\n",
    "print(\"LR 5-Fold CV F1 Scores:\", cv_scores)\n",
    "print(\"LR Mean F1 Score: {:.4f}\".format(cv_scores.mean()))\n",
    "\n",
    "# --- Train ---\n",
    "lr.fit(X_train_scaled, y_train)\n",
    "\n",
    "# --- Predict on Validation Set ---\n",
    "y_pred_val = lr.predict(X_val_scaled)\n",
    "\n",
    "# --- Evaluation ---\n",
    "print(\"\\nClassification Report (Validation Set):\")\n",
    "print(classification_report(y_val, y_pred_val))\n",
    "\n",
    "# --- Confusion Matrix ---\n",
    "cm = confusion_matrix(y_val, y_pred_val, labels=labels, normalize='true')\n",
    "plt.figure(figsize=(8, 6))\n",
    "sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',\n",
    "            xticklabels=[f\"Stage {l}\" for l in labels],\n",
    "            yticklabels=[f\"Stage {l}\" for l in labels])\n",
    "plt.title(\"FD002 – Logistic Regression Confusion Matrix (Validation Set)\")\n",
    "plt.xlabel(\"Predicted\")\n",
    "plt.ylabel(\"Actual\")\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# --- Save Model ---\n",
    "model_path = os.path.join(MODELS_DIR, 'lr_classifier_fd002.pkl')\n",
    "joblib.dump(lr, model_path)\n",
    "print(f\"Model saved to: {model_path}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 8: Train & Evaluate XGBoost Classifier"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# --- XGBoost Model ---\n",
    "xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=RANDOM_STATE)\n",
    "\n",
    "# --- Cross-Validation ---\n",
    "cv_scores = cross_val_score(xgb, X_train_scaled, y_train, cv=cv, scoring='f1_weighted')\n",
    "print(\"XGBoost 5-Fold CV F1 Scores:\", cv_scores)\n",
    "print(\"XGBoost Mean F1 Score: {:.4f}\".format(cv_scores.mean()))\n",
    "\n",
    "# --- Train ---\n",
    "xgb.fit(X_train_scaled, y_train)\n",
    "\n",
    "# --- Predict on Validation Set ---\n",
    "y_pred_val = xgb.predict(X_val_scaled)\n",
    "\n",
    "# --- Evaluation ---\n",
    "print(\"\\nClassification Report (Validation Set):\")\n",
    "print(classification_report(y_val, y_pred_val))\n",
    "\n",
    "# --- Confusion Matrix ---\n",
    "cm = confusion_matrix(y_val, y_pred_val, labels=labels, normalize='true')\n",
    "plt.figure(figsize=(8, 6))\n",
    "sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',\n",
    "            xticklabels=[f\"Stage {l}\" for l in labels],\n",
    "            yticklabels=[f\"Stage {l}\" for l in labels])\n",
    "plt.title(\"FD002 – XGBoost Confusion Matrix (Validation Set)\")\n",
    "plt.xlabel(\"Predicted\")\n",
    "plt.ylabel(\"Actual\")\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "\n",
    "# --- Save Model ---\n",
    "model_path = os.path.join(MODELS_DIR, 'xgb_classifier_fd002.pkl')\n",
    "joblib.dump(xgb, model_path)\n",
    "print(f\"Model saved to: {model_path}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 9: Test Set Evaluation (Best Model - SVM)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load the best model (SVM in this case)\n",
    "best_model = joblib.load(os.path.join(MODELS_DIR, 'svm_classifier_fd002.pkl'))\n",
    "\n",
    "# Predict on test set\n",
    "y_pred_test = best_model.predict(X_test_scaled)\n",
    "\n",
    "# Evaluation\n",
    "print(\"\\nClassification Report (Test Set):\")\n",
    "print(classification_report(y_test, y_pred_test))\n",
    "\n",
    "# Confusion Matrix\n",
    "cm_test = confusion_matrix(y_test, y_pred_test, labels=labels, normalize='true')\n",
    "plt.figure(figsize=(8, 6))\n",
    "sns.heatmap(cm_test, annot=True, fmt='.2f', cmap='Blues',\n",
    "            xticklabels=[f\"Stage {l}\" for l in labels],\n",
    "            yticklabels=[f\"Stage {l}\" for l in labels])\n",
    "plt.title(\"FD002 – SVM Confusion Matrix (Test Set)\")\n",
    "plt.xlabel(\"Predicted\")\n",
    "plt.ylabel(\"Actual\")\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 10: Save Predictions and Misclassified Samples"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Add predictions to test dataframe\n",
    "test_df = pd.DataFrame(X_test_scaled, columns=selected_features)\n",
    "test_df['y_true'] = y_test.values\n",
    "test_df['y_pred'] = y_pred_test\n",
    "\n",
    "# Save predictions and misclassified samples\n",
    "test_df.to_csv(os.path.join(CLASSIF_REPORT_DIR, 'predictions_test_fd002.csv'), index=False)\n",
    "misclassified = test_df[test_df['y_true'] != test_df['y_pred']]\n",
    "misclassified.to_csv(os.path.join(CLASSIF_REPORT_DIR, 'misclassified_test_fd002.csv'), index=False)\n",
    "print(\"✅ Test predictions and misclassified samples saved.\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.5"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}